# Exploring HyperSpy Tools to analyze the EDX dataset

In [1]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from EDX import *
from utils import *
from utils_noise import *
import pickle
import copy
import time
import humanfriendly
from skimage.restoration import estimate_sigma
from sklearn.decomposition import PCA
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
import random
import humanfriendly
import hyperspy.api as hs
import exspy

device = torch.device("mps")
print(device)

%load_ext autoreload
%autoreload 2

mps


## From Hyperspy documentation

### Download EDX data 
- Moved to data folder in parent dir).
- Useful to use as a sanity check later, to compare against out own HSIs.

In [ ]:
"""
from urllib.request import urlretrieve, urlopen
from zipfile import ZipFile
files = urlretrieve("https://www.dropbox.com/s/s7cx92mfh2zvt3x/"
                    "HyperSpy_demos_EDX_SEM_files.zip?raw=1",
                    "./HyperSpy_demos_EDX_SEM_files.zip")
with ZipFile("HyperSpy_demos_EDX_SEM_files.zip") as z:
    z.extractall()

"""


#### Also an option: load datasets directly from HS for similar reasons

In [ ]:
"""
si = hs.load("../data/HyperSpyData/Ni_superalloy_010.rpl").as_signal1D(0)
plt.imshow(si.data[:,:,500],cmap='gray')
"""

### Load own data

#### First, this loads presaved tiles, which are instances of the EM-EDX class. Skip this for now.

In [ ]:
# Load aligned dataset
with open('../preprocessing_basic/results/preprocessed_edx/20260119_132514_tile_unaligned_20.pkl', 'rb') as file:
    tile_20 = pickle.load(file)

tile_20.apply("crop", parameters={"crop_idx": (slice(50,tile_20_aligned.EDX_dim[0]-50), slice(50,tile_20_aligned.EDX_dim[1]-50), slice(None))})

#### Let's take a step back and load the HSI from HS again, and see if there are others options prior to making it into a numpy array and saving it into an EM-EDX instance
- First, we added the return_dict option which return the whole EDX dictionary from the velox EMD instead of just the numpy array
- Also, the content of the EMD is printed, including the fitted elemental maps

In [2]:
file_path = "../data/EMD/EDXdataset.emd"
edx, haadf, xray_energies = load_EDX(file_path, first_frame=0, last_frame=100, sum_frames=True, haadf_last_frame= True, return_dict=True)
#ss = edx_unaligned_20(add_powerlaw=True)
#ss.remove_background(signal_range=(100, 200))
#ss.plot()


WARNING | RosettaSciIO | The file contains only one spectrum stream (rsciio.emd._emd_velox:590)
[<EDSTEMSpectrum, title: EDS, dimensions: (|4096)>,
 <Signal2D, title: O, dimensions: (|2048, 2048)>,
 <Signal2D, title: Os, dimensions: (|2048, 2048)>,
 <Signal2D, title: HAADF, dimensions: (100|2048, 2048)>,
 <Signal2D, title: N, dimensions: (|2048, 2048)>,
 <Signal2D, title: S, dimensions: (|2048, 2048)>,
 <Signal2D, title: C, dimensions: (|2048, 2048)>,
 <Signal2D, title: Fe, dimensions: (|2048, 2048)>,
 <Signal2D, title: P, dimensions: (|2048, 2048)>,
 <EDSTEMSpectrum, title: EDS, dimensions: (2048, 2048|4096)>]


### Metadata

In [ ]:
edx.metadata

### More metadata

In [ ]:
edx.original_metadata

### Instrument metadata

In [ ]:
edx.metadata.Acquisition_instrument.TEM

### Axes manager

In [ ]:
edx.axes_manager

### Sample metadata/description
- adding elements and lines. This requires the beam energy to be set, which it is by velox, but if not, then set it with `edx.set_microscope_parameters(beam_energy=80)`
- some elements were set (for this sample), but maybe it's useful to add more (or all reasonable/expected ones) in general.

In [ ]:
edx.add_lines()
edx.metadata.Sample

#### Add more elements

In [ ]:
edx.add_elements(['C','N', 'O', 'P', 'S', 'Cl', 'Ca', 'Cu', 'Al', 'Fe', 'Os', 'Nd'])
edx.add_lines()
edx.metadata.Sample

#### Plotting

In [ ]:
edx.plot(True)

In [ ]:
edx.get_lines_intensity(['Na_Ka'], plot_result=True)

### EDX curve fitting

### fit model

In [ ]:
m = edx.inav[:-1:30,:-1:30].create_model()

#m.print_current_values()

start = time.perf_counter()
m.multifit()
elapsed = time.perf_counter() - start
print("Step 1 time:", humanfriendly.format_timespan(elapsed))

#m.fit_background()
#elapsed = time.perf_counter() - start
#print("Step 2 time:", humanfriendly.format_timespan(elapsed))

#m.calibrate_energy_axis(calibrate='resolution')
#elapsed = time.perf_counter() - start
#print("Step 2 time:", humanfriendly.format_timespan(elapsed))

In [ ]:
#fitted = m.as_signal()

#fitted = m.as_signal(component_list=[c for c in m.components if all(p.map['is_set'].all() for p in c.parameters)])

#bg_comp = [c for c in m.components if 'background' in c.name.lower()][0]

#bg = bg_comp.as_signal()

#peaks = sum(component.as_signal() for component in m.components if component is not background)

#bg = m.components.background_order_6.as_signal()


bg = m.as_signal(component_list=['background_order_6'])



In [ ]:
%matplotlib qt
f, ax = plt.subplots(1,3)
ax[0].plot(edx.data.mean(axis=(0,1)),'b')
ax[1].plot(bg.data.mean(axis=(0,1)),'r')
ax[2].plot(bg.data.mean(axis=(0,1)),'r')
ax[2].plot(edx.data.mean(axis=(0,1)),'b')

In [ ]:
### s =  exspy.data.EDS_TEM_FePt_nanoparticles().isig[5.:13.]
s = hs.signals.Signal1D(tile_20.EDX)

s.set_signal_type("EDS_TEM")
s.axes_manager[2].name = 'Energy'
s.axes_manager['Energy'].units = 'keV'
s.axes_manager['Energy'].scale = 1
s.axes_manager['Energy'].offset = 0 
s.axes_manager['Energy'].axis = tile_20_aligned.xray_energies

s.axes_manager[0].name = 'X'
s.axes_manager['X'].scale = 2.5  
s.axes_manager['X'].units = "nm"
s.axes_manager['X'].offset = 0  

s.axes_manager[1].name = 'Y'
s.axes_manager['Y'].scale = 2.5  
s.axes_manager['Y'].units = "nm"
s.axes_manager['Y'].offset = 0  


#s.add_elements(['C','N', 'O', 'P', 'S', 'Cl', 'Ca', 'Cu', 'Al', 'Fe', 'Os', 'Nd'])
#s.add_elements(['N', 'O', 'P', 'S', 'Cl', 'Ca', 'Cu', 'Al', 'Fe', 'Os', 'Nd'])
#s.add_elements(['Al', 'Fe'])

#s.add_lines([['C', 'Ka'],['N', 'Ka'],['O', 'Ka'],['P', 'Ka'],['S', 'Ka'],['Cl', 'Ka'],['Ca', 'Ka'],['Cu', 'Ka'],
#              ['Cu', 'La'],['Al', 'Ka'],['Fe', 'Ka'],['Fe', 'La'],['Os', 'Ma'],['Os', 'La'],['Nd','La']])


#bw = s.estimate_background_windows(line_width=[2, 2])
#s.plot()


#s.get_lines_intensity(background_windows=bw, plot_result=True)




# Add lines automatically
s.set_microscope_parameters(beam_energy=10)
s.add_lines()
s.metadata.Sample

In [19]:
edx_bg_removed = copy.deepcopy(edx)


In [15]:
print(edx_bg_removed)
edx_bg_removed = edx_bg_removed.rebin(scale=[1,1,8])
edx_bg_removed = edx_bg_removed.inav[:200,:200]
print(edx_bg_removed)
edx_bg_removed.axes_manager

<EDSTEMSpectrum, title: EDS, dimensions: (200, 200|64)>
<EDSTEMSpectrum, title: EDS, dimensions: (200, 200|8)>


Navigation axis name,size,index,offset,scale,units
x,200,0,-2.559999999999999,0.002499999999999999,µm
y,200,0,-2.559999999999999,0.002499999999999999,µm
Signal axis name,size,,offset,scale,units
X-ray energy,8,,0.7976939850000002,2.56,keV


In [ ]:
start = time.perf_counter()
edx_bg_removed = edx_bg_removed.remove_background(signal_range=(1,1.25),background_type="Power law",fast=False)
elapsed = time.perf_counter() - start
print("Background removal time:", humanfriendly.format_timespan(elapsed))

WARNING | Hyperspy | Power-law parameter estimation failed because of a "divide-by-zero" error. (hyperspy._components.power_law:166)


  0%|          | 0/40000 [00:00<?, ?it/s]

WARNING | Hyperspy | Covariance of the parameters could not be estimated. Estimated parameter standard deviations will be np.nan. (hyperspy.model:1743)
WARNING | Hyperspy | `m.fit()` did not exit successfully. Reason: Number of calls to function has reached maxfev = 600. (hyperspy.model:2208)
WARNING | Hyperspy | Covariance of the parameters could not be estimated. Estimated parameter standard deviations will be np.nan. (hyperspy.model:1743)
WARNING | Hyperspy | `m.fit()` did not exit successfully. Reason: Number of calls to function has reached maxfev = 600. (hyperspy.model:2208)
WARNING | Hyperspy | Covariance of the parameters could not be estimated. Estimated parameter standard deviations will be np.nan. (hyperspy.model:1743)
WARNING | Hyperspy | Covariance of the parameters could not be estimated. Estimated parameter standard deviations will be np.nan. (hyperspy.model:1743)
WARNING | Hyperspy | Covariance of the parameters could not be estimated. Estimated parameter standard devia

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



WARNING | Hyperspy | Covariance of the parameters could not be estimated. Estimated parameter standard deviations will be np.nan. (hyperspy.model:1743)
WARNING | Hyperspy | Covariance of the parameters could not be estimated. Estimated parameter standard deviations will be np.nan. (hyperspy.model:1743)
WARNING | Hyperspy | Covariance of the parameters could not be estimated. Estimated parameter standard deviations will be np.nan. (hyperspy.model:1743)
WARNING | Hyperspy | Covariance of the parameters could not be estimated. Estimated parameter standard deviations will be np.nan. (hyperspy.model:1743)
WARNING | Hyperspy | Covariance of the parameters could not be estimated. Estimated parameter standard deviations will be np.nan. (hyperspy.model:1743)
WARNING | Hyperspy | Covariance of the parameters could not be estimated. Estimated parameter standard deviations will be np.nan. (hyperspy.model:1743)
WARNING | Hyperspy | Covariance of the parameters could not be estimated. Estimated para

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



WARNING | Hyperspy | Covariance of the parameters could not be estimated. Estimated parameter standard deviations will be np.nan. (hyperspy.model:1743)
WARNING | Hyperspy | Covariance of the parameters could not be estimated. Estimated parameter standard deviations will be np.nan. (hyperspy.model:1743)
WARNING | Hyperspy | Covariance of the parameters could not be estimated. Estimated parameter standard deviations will be np.nan. (hyperspy.model:1743)
WARNING | Hyperspy | Covariance of the parameters could not be estimated. Estimated parameter standard deviations will be np.nan. (hyperspy.model:1743)
WARNING | Hyperspy | Covariance of the parameters could not be estimated. Estimated parameter standard deviations will be np.nan. (hyperspy.model:1743)
WARNING | Hyperspy | Covariance of the parameters could not be estimated. Estimated parameter standard deviations will be np.nan. (hyperspy.model:1743)
WARNING | Hyperspy | Covariance of the parameters could not be estimated. Estimated para

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



WARNING | Hyperspy | Covariance of the parameters could not be estimated. Estimated parameter standard deviations will be np.nan. (hyperspy.model:1743)
WARNING | Hyperspy | Covariance of the parameters could not be estimated. Estimated parameter standard deviations will be np.nan. (hyperspy.model:1743)
WARNING | Hyperspy | Covariance of the parameters could not be estimated. Estimated parameter standard deviations will be np.nan. (hyperspy.model:1743)
WARNING | Hyperspy | Covariance of the parameters could not be estimated. Estimated parameter standard deviations will be np.nan. (hyperspy.model:1743)
WARNING | Hyperspy | Covariance of the parameters could not be estimated. Estimated parameter standard deviations will be np.nan. (hyperspy.model:1743)
WARNING | Hyperspy | Covariance of the parameters could not be estimated. Estimated parameter standard deviations will be np.nan. (hyperspy.model:1743)
WARNING | Hyperspy | Covariance of the parameters could not be estimated. Estimated para

In [6]:
print(edx)
edx = edx.rebin(scale=[1,1,8])
edx = edx.inav[:200,:200]
print(edx)
edx.axes_manager

<EDSTEMSpectrum, title: EDS, dimensions: (2048, 2048|4096)>
<EDSTEMSpectrum, title: EDS, dimensions: (200, 200|512)>


Navigation axis name,size,index,offset,scale,units
x,200,0,-2.559999999999999,0.002499999999999999,µm
y,200,0,-2.559999999999999,0.002499999999999999,µm
Signal axis name,size,,offset,scale,units
X-ray energy,512,,-0.46230601499999996,0.04,keV


In [21]:
%matplotlib qt
f, ax = plt.subplots(1,3)
ax[0].plot(edx.data.sum(axis=(0,1)),'b')
ax[1].plot(edx_bg_removed.data.sum(axis=(0,1)),'r')
ax[2].plot(edx.data.sum(axis=(0,1)),'b')  
ax[2].plot(edx_bg_removed.data.sum(axis=(0,1)),'r')  

/Users/aj/Desktop/work/virtualenvs/edx_new/lib/python3.11/site-packages/numpy/_core/_methods.py:52: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


In [ ]:
%matplotlib qt
plt.plot(edx.data.sum(axis=(0,1)))